# A Simple Diffusion Model on the MNIST Data Set using JAX / FLAX

This notebook is intended to store a simple implementation of a [diffusion model](https://en.wikipedia.org/wiki/Diffusion_model) as well as its training process on the [MNIST data set](https://www.tensorflow.org/datasets/catalog/mnist) using the Python libraries [JAX](https://docs.jax.dev/en/latest/index.html) and [FLAX](https://flax.readthedocs.io/en/stable/).

<center>
    <img src=imgs/jax.webp>
</center>

Let us first import the necessary data set. In this project, we are going to use the $(28 \times 28)$ MNIST images that are contained in the `tensorflow` package.

First of all, we are going to import the necessary modules.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

2026-01-30 13:22:17.509775: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-30 13:22:17.537374: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Loading the MNIST Data

This section deals with loading the MNIST data set from the `tensorflow_datasets` package.

In [2]:
# Load the actual data
with tf.device("/cpu:0"):
    train_ds = tfds.load(name="mnist", split="train")
    X = np.array([batch["image"] for batch in train_ds.as_numpy_iterator()]).astype(np.float16)
    y = np.array([batch["label"] for batch in train_ds.as_numpy_iterator()]).astype(np.int8)

# Normalize to interval [-1, 1]
X = -1 + X * 2 / 255.0

# Resize from (28 x 28) to (32 x 32)
b, _, _, c = X.shape
X = tf.image.resize(X, size=(32, 32), method="nearest").numpy()

X.shape, y.shape, X.max(), X.min()

I0000 00:00:1769775744.713611 3727940 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13675 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:05:00.0, compute capability: 8.9
2026-01-30 13:22:26.314643: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-01-30 13:22:32.926296: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-01-30 13:22:39.121136: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-01-30 13:22:39.615900: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


((60000, 32, 32, 1), (60000,), np.float16(1.0), np.float16(-1.0))

We have resized the images from $(28 \times 28)$ pixels to $(32 \times 32)$ pixels in order to make the downsampling steps more convenient later.

Let us display some of the images.

In [3]:
import ipywidgets

def plot(index):
    image = X[index]
    plt.imshow(image, cmap="gray")
    plt.title(f"Image of class {y[index]}")

ipywidgets.interact(plot, index=ipywidgets.IntSlider(min=0, max=X.shape[0], step=1))

interactive(children=(IntSlider(value=0, description='index', max=60000), Output()), _dom_classes=('widget-int…

<function __main__.plot(index)>

## Implementing the Diffusion Model

**TODO**